<a href="https://colab.research.google.com/github/ravindrakumarvakkalagadda-tech/Physics-Informed-ML-for-Transient-Temperature-Prediction-in-Moving-Heat-Source-Welding/blob/main/HeatTransferAnalysisInWelding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
"""
A Physics-Informed Machine Learning Framework for Predicting Transient Temperature Distribution in Moving Heat Source Welding Processes
Reference: 1. Rocha, E. J. F., Antonino, T. de S., Guimarães, P. B., Ferreira, R. A. S., Barbosa, J. M. A., & Rohatgi, J. (2018). Modeling of the temperature field generated by the deposition of weld bead on a steel butt joint by FEM techniques and thermographic images. Materials Research, 21(3), e20160796. https://doi.org/10.1590/1980-5373-MR-2016-0796)

Method:
- Cell-centred explicit finite-volume Method conduction solver.
- Moving Goldak double-ellipsoidal volumetric heat source and optional uniform surface-flux comparison model.
- linear temperature-dependent thermal properties k(T), alpha(T), cp(T), and supplied rho(T).
- Material-properties obtained fromt he reference in the range of 0-650 C;
- Radiation related information extracted from the manuscript.
- The bottom face is in full contact with a steel support table and uses a contact-conductance boundary condition.

"""

from dataclasses import dataclass
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------- INPUT PARAMETERS -----------------------------

@dataclass
class Config:
    # Specimen dimensions (manuscript: 150 x 80 x 7.5 mm)
    Lx: float = 0.150       # m
    Ly: float = 0.080       # m
    Lz: float = 0.0075      # m

    # Refined XYZ mesh used in the latest completed simulation
    dx: float = 0.003      # m (3 mm)
    dy: float = 0.001      # m (1 mm)
    dz: float = 0.0015     # m (1.5 mm)

    # Welding parameters: manuscript Table 2
    current: float = 192.0         # A
    voltage: float = 18.4          # V
    efficiency: float = 0.85       # -
    speed: float = 0.00278         # m/s
    weld_length: float = 0.100     # m; bead length (plate length remains 150 mm)
    x_start: float = 0.025         # m; 25 mm unwelded margin at start
    y_track: float = 0.040         # m; specimen centreline
    weld_time: float = 0.0         # s; retained for reference, run duration is calculated

    # Goldak parameters: manuscript Table 4
    b: float = 0.004472      # m
    c: float = 0.00605       # m
    af: float = 0.003025     # m
    ar: float = 0.0121       # m
    ff: float = 0.4
    fr: float = 1.6

    # Select "uniform_surface" for the comparative surface-flux model,
    # or "goldak" for the original volumetric model.
    source_model: str = "goldak"
    footprint_length: float = 0.010  # m, along welding direction
    footprint_width: float = 0.008   # m, across welding direction

    # Thermal boundary conditions
    T_ambient_C: float = 27.0
    emissivity: float = 0.91
    sigma: float = 5.670374419e-8  # W/(m2 K4), physical constant
    h_convection: float = 6.0     # W/(m2 K), assumed exposed-surface coefficient

    # Bottom contact with a fully supporting steel table.
    # IMPORTANT: h_contact is a provisional sensitivity-study input, not measured.
    h_contact: float = 3000.0      # W/(m2 K), provisional contact conductance
    T_table_C: float = 27.0       # deg C, fixed-temperature table approximation

    # Numerical controls
    dt_requested: float = 0.0025  # s; further limited by explicit stability
    output_interval: float = 1.0  # s
    max_temperature_rise_per_step: float = 5.0  # K; source-heating safeguard
    output_dir: str = "welding_results_steel_table_contact"


CFG = Config()

# Temperature-dependent material properties supplied in the user's latest table.
# The table currently covers 0-650 C only. Values are not smoothed.
MAT_T = np.array([0, 100, 200, 300, 400, 500, 550, 600, 650], dtype=float)
K_TABLE = np.array(
    [51.062, 51.062, 47.324, 43.840, 40.424, 36.950, 33.546, 26.461, 18.019],
    dtype=float
)  # W/(m K)
ALPHA_TABLE = np.array(
    [14.000e-6, 13.150e-6, 11.980e-6, 10.870e-6, 9.670e-6,
     8.237e-6, 7.358e-6, 5.991e-6, 3.647e-6],
    dtype=float
)  # m2/s
CP_TABLE = np.array(
    [571.235, 571.235, 571.235, 571.235, 614.841, 668.617,
     730.831, 780.774, 868.623],
    dtype=float
)  # J/(kg K)
RHO_TABLE = np.array(
    [6384.912889, 6797.625890, 6915.280782, 7060.349375, 6799.077490,
     6709.157113, 6238.267569, 5656.940234, 5688.052513],
    dtype=float
)  # kg/m3; supplied explicitly by the user


def material_properties(T_C):

    k = np.interp(T_C, MAT_T, K_TABLE)
    alpha = np.interp(T_C, MAT_T, ALPHA_TABLE)
    cp = np.interp(T_C, MAT_T, CP_TABLE)
    rho = np.interp(T_C, MAT_T, RHO_TABLE)
    return k, cp, alpha, rho


def material_table_diagnostics():
    """Return the supplied properties."""
    rho_from_identity = K_TABLE / (ALPHA_TABLE * CP_TABLE)
    return pd.DataFrame({
        "temperature_C": MAT_T,
        "k_W_mK": K_TABLE,
        "alpha_m2_s": ALPHA_TABLE,
        "cp_J_kgK": CP_TABLE,
        "density_supplied_kg_m3": RHO_TABLE,
        "density_from_k_alpha_cp_kg_m3": rho_from_identity,
        "density_difference_kg_m3": RHO_TABLE - rho_from_identity,
    })


def make_grid(cfg):
    nx = int(round(cfg.Lx / cfg.dx))
    ny = int(round(cfg.Ly / cfg.dy))
    nz = int(round(cfg.Lz / cfg.dz))
    dx, dy, dz = cfg.Lx/nx, cfg.Ly/ny, cfg.Lz/nz
    x = (np.arange(nx) + 0.5) * dx
    y = (np.arange(ny) + 0.5) * dy
    z = (np.arange(nz) + 0.5) * dz
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
    return x, y, z, X, Y, Z, dx, dy, dz


def goldak_source(X, Y, Z, t, cfg):
    """
    Goldak double ellipsoid, volumetric source W/m3.
    x-axis is the travel direction; source moves in +x.
    Front is ahead of torch (xi >= 0), rear is behind torch (xi < 0).
    The discrete source is renormalised so sum(q*cell_volume) = eta*V*I.
    """
    power = cfg.efficiency * cfg.voltage * cfg.current
    xs = cfg.x_start + cfg.speed * t
    xi = X - xs
    eta_y = Y - cfg.y_track

    q = np.zeros_like(X)
    front = xi >= 0.0
    rear = ~front

    # Goldak Gaussian shape factors. The factors ff/fr describe the
    # relative front/rear distribution. Discrete normalisation below
    # guarantees the specified total net power in this finite domain.
    pref_f = 6.0*np.sqrt(3.0)*cfg.ff*power / (
        np.pi**1.5 * cfg.af * cfg.b * cfg.c
    )
    pref_r = 6.0*np.sqrt(3.0)*cfg.fr*power / (
        np.pi**1.5 * cfg.ar * cfg.b * cfg.c
    )

    q[front] = pref_f * np.exp(
        -3.0*(xi[front]**2/cfg.af**2
              + eta_y[front]**2/cfg.b**2
              + Z[front]**2/cfg.c**2)
    )
    q[rear] = pref_r * np.exp(
        -3.0*(xi[rear]**2/cfg.ar**2
              + eta_y[rear]**2/cfg.b**2
              + Z[rear]**2/cfg.c**2)
    )

    # Numerical energy normalisation: prevents finite-grid truncation or
    # front/rear convention ambiguity from changing total applied power.
    cell_volume = (cfg.Lx / X.shape[0]) * (cfg.Ly / X.shape[1]) * (cfg.Lz / X.shape[2])
    integrated = np.sum(q) * cell_volume
    if integrated > 0:
        q *= power / integrated
    return q, xs, power


def uniform_surface_source(X, Y, Z, t, cfg, dx, dy, dz):
    """Apply a moving uniform heat flux to the top surface using exact overlap areas.

    Returns a volumetric-equivalent source (W/m3) in the first layer of cells.
    For each top-face cell, the heat deposited is q_flux * overlap_area.
    Dividing that power by the cell volume gives the source term used in the
    cell-centred energy equation. The sum over all cells is exactly eta*V*I
    whenever the complete footprint lies inside the plate.
    """
    power = cfg.efficiency * cfg.voltage * cfg.current  # W entering the plate
    xs = cfg.x_start + cfg.speed * t

    # Fixed nominal footprint and flux; efficiency is applied exactly once.
    footprint_area = cfg.footprint_length * cfg.footprint_width
    q_flux = power / footprint_area  # W/m2

    # Cell-face edges in the x-y plane. x and y are cell-centre coordinates.
    x_centres = X[:, 0, 0]
    y_centres = Y[0, :, 0]
    x_left = x_centres - dx/2.0
    x_right = x_centres + dx/2.0
    y_left = y_centres - dy/2.0
    y_right = y_centres + dy/2.0

    # Moving rectangular footprint boundaries.
    src_x_left = xs - cfg.footprint_length/2.0
    src_x_right = xs + cfg.footprint_length/2.0
    src_y_left = cfg.y_track - cfg.footprint_width/2.0
    src_y_right = cfg.y_track + cfg.footprint_width/2.0

    # Exact geometric overlap of each surface cell with the source rectangle.
    overlap_x = np.maximum(
        0.0, np.minimum(x_right, src_x_right) - np.maximum(x_left, src_x_left)
    )
    overlap_y = np.maximum(
        0.0, np.minimum(y_right, src_y_right) - np.maximum(y_left, src_y_left)
    )
    overlap_area = overlap_x[:, None] * overlap_y[None, :]

    covered_area = float(np.sum(overlap_area))
    if covered_area <= 0.0:
        raise RuntimeError("Moving heat-source footprint does not overlap the plate surface.")

    # Do not silently renormalise a truncated footprint at a plate edge.
    # The selected weld path should keep the full footprint inside the domain.
    area_tolerance = max(1e-12, footprint_area*1e-9)
    if abs(covered_area - footprint_area) > area_tolerance:
        raise RuntimeError(
            f"Heat-source footprint is partly outside the plate: "
            f"covered area={covered_area:.8e} m2, nominal area={footprint_area:.8e} m2. "
            "Check x_start, y_track, weld path and specimen dimensions."
        )

    qvol = np.zeros_like(X)
    # q_flux*overlap_area is power into each top-face cell (W).
    # Divide by dx*dy*dz to obtain W/m3 in the top boundary cell.
    qvol[:, :, 0] = q_flux * overlap_area / (dx*dy*dz)

    # Numerical power check: sum(qvol*cell_volume) must equal eta*V*I.
    applied_power = float(np.sum(qvol[:, :, 0]) * dx*dy*dz)
    if not np.isclose(applied_power, power, rtol=1e-10, atol=1e-8):
        raise RuntimeError(
            f"Heat-input check failed: applied {applied_power:.8f} W, expected {power:.8f} W."
        )

    n_cells = int(np.count_nonzero(overlap_area))
    return qvol, xs, power, q_flux, covered_area, n_cells


def source_field(X, Y, Z, t, cfg, dx, dy, dz):
    """Return volumetric-equivalent source field and diagnostic metadata."""
    if cfg.source_model == "uniform_surface":
        q, xs, power, q_flux, area, n_cells = uniform_surface_source(
            X, Y, Z, t, cfg, dx, dy, dz
        )
        return q, xs, power, {
            "surface_flux_W_m2": q_flux,
            "covered_footprint_area_m2": area,
            "nominal_footprint_area_m2": cfg.footprint_length*cfg.footprint_width,
            "footprint_cells": n_cells
        }
    elif cfg.source_model == "goldak":
        q, xs, power = goldak_source(X, Y, Z, t, cfg)
        return q, xs, power, {
            "surface_flux_W_m2": None,
            "discrete_footprint_area_m2": None,
            "footprint_cells": None
        }
    else:
        raise ValueError("source_model must be 'uniform_surface' or 'goldak'.")


def conductive_divergence(T, k, dx, dy, dz):
    """Compute div(k grad T) with harmonic face conductivity and zero conductive flux at outer faces."""
    nx, ny, nz = T.shape
    div = np.zeros_like(T)

    # x-direction internal faces
    kf = 2*k[1:,:,:]*k[:-1,:,:] / (k[1:,:,:] + k[:-1,:,:] + 1e-30)
    gx = kf * (T[1:,:,:] - T[:-1,:,:]) / dx
    div[:-1,:,:] += gx/dx
    div[1:,:,:] -= gx/dx

    # y-direction
    kf = 2*k[:,1:,:]*k[:,:-1,:] / (k[:,1:,:] + k[:,:-1,:] + 1e-30)
    gy = kf * (T[:,1:,:] - T[:,:-1,:]) / dy
    div[:,:-1,:] += gy/dy
    div[:,1:,:] -= gy/dy

    # z-direction
    kf = 2*k[:,:,1:]*k[:,:,:-1] / (k[:,:,1:] + k[:,:,:-1] + 1e-30)
    gz = kf * (T[:,:,1:] - T[:,:,:-1]) / dz
    div[:,:,:-1] += gz/dz
    div[:,:,1:] -= gz/dz

    return div


def boundary_loss(T, k, cfg, dx, dy, dz):
    """Boundary heat losses as W/m3 in boundary cells.

    Exposed top and side faces use convection + radiation. The bottom face
    uses conduction through the half-cell in series with the plate/table
    contact resistance. This gives a face heat flux and face temperature
    consistent with the cell-centred finite-volume representation.
    """
    TinfK = cfg.T_ambient_C + 273.15
    TK = np.maximum(T + 273.15, 1.0)
    loss = np.zeros_like(T)

    # Top face: convection + radiation to ambient.
    qtop = (cfg.h_convection*(T[:, :, 0] - cfg.T_ambient_C)
            + cfg.emissivity*cfg.sigma*(TK[:, :, 0]**4 - TinfK**4))
    loss[:, :, 0] += qtop/dz

    # Bottom face: half-cell conduction resistance + contact resistance.
    # q = (T_cell - T_table) / (dz/(2*k) + 1/h_contact)
    k_bottom = np.maximum(k[:, :, -1], 1.0e-12)
    h_eff = 1.0 / (1.0/cfg.h_contact + dz/(2.0*k_bottom))
    q_contact = h_eff*(T[:, :, -1] - cfg.T_table_C)
    loss[:, :, -1] += q_contact/dz

    # Four side faces: convection + radiation to ambient.
    qx0 = (cfg.h_convection*(T[0, :, :] - cfg.T_ambient_C)
           + cfg.emissivity*cfg.sigma*(TK[0, :, :]**4 - TinfK**4))
    qx1 = (cfg.h_convection*(T[-1, :, :] - cfg.T_ambient_C)
           + cfg.emissivity*cfg.sigma*(TK[-1, :, :]**4 - TinfK**4))
    loss[0, :, :] += qx0/dx
    loss[-1, :, :] += qx1/dx

    qy0 = (cfg.h_convection*(T[:, 0, :] - cfg.T_ambient_C)
           + cfg.emissivity*cfg.sigma*(TK[:, 0, :]**4 - TinfK**4))
    qy1 = (cfg.h_convection*(T[:, -1, :] - cfg.T_ambient_C)
           + cfg.emissivity*cfg.sigma*(TK[:, -1, :]**4 - TinfK**4))
    loss[:, 0, :] += qy0/dy
    loss[:, -1, :] += qy1/dy
    return loss


def reconstruct_backside_face_temperature(T, k, cfg, dz):
    """Reconstruct the physical bottom-face temperature from the last cell.

    The bottom face is at z=Lz. The last cell centre is dz/2 above it.
    The reconstruction satisfies the same series-resistance contact model
    used in boundary_loss().
    """
    T_cell = T[:, :, -1]
    k_cell = np.maximum(k[:, :, -1], 1.0e-12)
    h_eff = 1.0 / (1.0/cfg.h_contact + dz/(2.0*k_cell))
    q_contact = h_eff*(T_cell - cfg.T_table_C)
    T_face = T_cell - q_contact*dz/(2.0*k_cell)
    return T_face


def sample_bilinear_xy(field, x, y, x_query, y_query):
    """Bilinearly sample a cell-centred x-y field at an in-domain location."""
    i0, i1, fx = _linear_bracket(x, x_query)
    j0, j1, fy = _linear_bracket(y, y_query)
    v00 = field[i0, j0]
    v10 = field[i1, j0]
    v01 = field[i0, j1]
    v11 = field[i1, j1]
    return float((1-fx)*(1-fy)*v00 + fx*(1-fy)*v10
                 + (1-fx)*fy*v01 + fx*fy*v11)


def stable_timestep(cfg, dx, dy, dz, X, Y, Z):
    """Conservative explicit diffusion limit and source-heating limit.

    The source limit is calculated from the DISCRETELY NORMALISED Goldak field,
    not the unnormalised analytical peak. This matters when the source is
    truncated by the finite computational domain.
    """
    # Conservative bounds based on the tabulated temperature-dependent properties.
    alpha_max = float(np.max(ALPHA_TABLE))
    volumetric_heat_capacity_min = float(np.min(RHO_TABLE * CP_TABLE))
    dt_diff = 0.35 / (2.0*alpha_max*(1/dx**2 + 1/dy**2 + 1/dz**2))
    q0, _, _, _ = source_field(X, Y, Z, 0.0, cfg, dx, dy, dz)
    q_peak_discrete = float(np.max(q0))
    dt_source = cfg.max_temperature_rise_per_step * volumetric_heat_capacity_min / q_peak_discrete
    return min(cfg.dt_requested, dt_diff, dt_source), dt_diff, dt_source


def _linear_bracket(grid, value):
    """Return indices and interpolation fraction for a monotonic 1-D grid."""
    if value < grid[0] or value > grid[-1]:
        raise ValueError(f"Probe coordinate {value} lies outside grid [{grid[0]}, {grid[-1]}].")
    hi = int(np.searchsorted(grid, value, side="left"))
    if hi == 0:
        return 0, 0, 0.0
    if hi >= len(grid):
        return len(grid)-1, len(grid)-1, 0.0
    lo = hi - 1
    frac = (value - grid[lo]) / (grid[hi] - grid[lo])
    return lo, hi, float(frac)


def sample_temperature_trilinear(T, x, y, z, x_query, y_query, z_query):
    """Trilinearly interpolate T at an exact physical (x,y,z) coordinate."""
    i0, i1, fx = _linear_bracket(x, x_query)
    j0, j1, fy = _linear_bracket(y, y_query)
    k0, k1, fz = _linear_bracket(z, z_query)

    c000 = T[i0, j0, k0]
    c100 = T[i1, j0, k0]
    c010 = T[i0, j1, k0]
    c110 = T[i1, j1, k0]
    c001 = T[i0, j0, k1]
    c101 = T[i1, j0, k1]
    c011 = T[i0, j1, k1]
    c111 = T[i1, j1, k1]

    c00 = c000*(1-fx) + c100*fx
    c10 = c010*(1-fx) + c110*fx
    c01 = c001*(1-fx) + c101*fx
    c11 = c011*(1-fx) + c111*fx
    c0 = c00*(1-fy) + c10*fy
    c1 = c01*(1-fy) + c11*fy
    return float(c0*(1-fz) + c1*fz)


def verify_interpolation_implementation():
    """Unit test: trilinear interpolation must reproduce a linear 3-D field."""
    xt = np.array([0.0, 1.0, 2.0])
    yt = np.array([0.0, 1.0, 2.0])
    zt = np.array([0.0, 1.0, 2.0])
    XX, YY, ZZ = np.meshgrid(xt, yt, zt, indexing="ij")
    test_field = 100.0 + 2.0*XX + 5.0*YY + 7.0*ZZ
    got = sample_temperature_trilinear(test_field, xt, yt, zt, 0.4, 0.8, 1.2)
    expected = 100.0 + 2.0*0.4 + 5.0*0.8 + 7.0*1.2
    if not np.isclose(got, expected, rtol=0.0, atol=1e-10):
        raise RuntimeError(f"Interpolation unit test failed: got {got}, expected {expected}")
    return True


def run_simulation(cfg=CFG):
    out = Path(cfg.output_dir)
    out.mkdir(parents=True, exist_ok=True)

    verify_interpolation_implementation()
    x, y, z, X, Y, Z, dx, dy, dz = make_grid(cfg)
    T = np.full(X.shape, cfg.T_ambient_C, dtype=np.float64)
    dt, dt_diff, dt_source = stable_timestep(cfg, dx, dy, dz, X, Y, Z)

    # The 150 mm plate has a 100 mm bead, beginning 25 mm from its end.
    # Therefore the bead ends at x = 125 mm, leaving a 25 mm end margin.
    weld_end_x = cfg.x_start + cfg.weld_length
    if cfg.x_start < 0 or weld_end_x > cfg.Lx:
        raise ValueError('Weld path must lie completely within the plate domain.')
    calculated_weld_time = cfg.weld_length / cfg.speed
    t_end = calculated_weld_time
    nsteps = int(np.ceil(t_end/dt))
    save_every = max(1, int(round(cfg.output_interval/dt)))

    # Probe locations: exact physical coordinates, sampled at several depths.
    # z=1.0 and 3.75 mm are subsurface planes; z=z[-1] is the rear-adjacent
    # cell-centre plane (not the exact geometric back surface at 7.5 mm).
    offsets_mm = np.array([0,1,2,3,5,7,8,10], dtype=float)
    probe_x_m = cfg.x_start + 0.066
    probe_y_m = cfg.y_track + offsets_mm/1000.0
    probe_depths_mm = [1.0, 3.75, float(z[-1]*1000.0), cfg.Lz*1000.0]
    probe_depths_m = [v/1000.0 for v in probe_depths_mm]

    probe_records = []
    snapshots = []
    energy_records = []
    cumulative_input_J = 0.0
    cumulative_boundary_loss_J = 0.0
    cumulative_storage_change_J = 0.0

    cell_vol = dx*dy*dz
    t = 0.0
    for step in range(nsteps + 1):
        if step % save_every == 0 or step == nsteps:
            for depth_mm, depth_m in zip(probe_depths_mm, probe_depths_m):
                for off, yq in zip(offsets_mm, probe_y_m):
                    if np.isclose(depth_mm, cfg.Lz*1000.0):
                        k_now, _, _, _ = material_properties(T)
                        T_back_face = reconstruct_backside_face_temperature(T, k_now, cfg, dz)
                        temp_probe = sample_bilinear_xy(
                            T_back_face, x, y, probe_x_m, float(yq)
                        )
                    else:
                        temp_probe = sample_temperature_trilinear(
                            T, x, y, z, probe_x_m, float(yq), float(depth_m)
                        )
                    probe_records.append([t, off, depth_mm, temp_probe])
            snapshots.append((t, T[:, :, 0].copy()))  # top-surface plane
            energy_residual_J = cumulative_storage_change_J - (
                cumulative_input_J - cumulative_boundary_loss_J
            )
            energy_records.append([
                t, float(T.max()), float(T.mean()),
                cumulative_input_J, cumulative_boundary_loss_J,
                cumulative_storage_change_J, energy_residual_J
            ])

        if step == nsteps:
            break

        k, cp, alpha, rho = material_properties(T)
        qvol, xs, net_power, source_info = source_field(
            X, Y, Z, t, cfg, dx, dy, dz
        )
        div_cond = conductive_divergence(T, k, dx, dy, dz)
        qloss = boundary_loss(T, k, cfg, dx, dy, dz)

        dTdt = (div_cond + qvol - qloss) / (rho*cp)
        T_new = T + dt*dTdt

        # Global discrete energy accounting. The conduction operator uses
        # conservative internal face fluxes and zero conductive flux at the
        # outer faces, so its domain-integrated contribution should cancel.
        applied_power_W = float(np.sum(qvol) * cell_vol)
        boundary_loss_W = float(np.sum(qloss) * cell_vol)
        storage_increment_J = float(np.sum(rho*cp*(T_new - T)) * cell_vol)
        cumulative_input_J += applied_power_W * dt
        cumulative_boundary_loss_J += boundary_loss_W * dt
        cumulative_storage_change_J += storage_increment_J
        T = T_new
        t += dt

        if not np.isfinite(T).all():
            raise FloatingPointError("Non-finite temperature detected. Reduce timestep or inspect inputs.")

    probes = pd.DataFrame(
        probe_records,
        columns=["time_s", "distance_from_weld_line_mm", "probe_depth_mm", "temperature_C"]
    )
    probes.to_csv(out/"probe_temperature_histories.csv", index=False)
    material_table_diagnostics().to_csv(out/"material_property_density_check.csv", index=False)

    # Check identical histories separately at each requested depth.
    for depth_mm in probe_depths_mm:
        subset = probes[probes.probe_depth_mm == depth_mm]
        wide = subset.pivot(index="time_s", columns="distance_from_weld_line_mm", values="temperature_C")
        identical_pairs = []
        for a_idx, a in enumerate(offsets_mm):
            for b in offsets_mm[a_idx+1:]:
                if np.array_equal(wide[a].to_numpy(), wide[b].to_numpy()):
                    identical_pairs.append([float(a), float(b)])
        if identical_pairs:
            print(f"WARNING: identical histories at depth {depth_mm:g} mm, offset pairs:", identical_pairs)
        else:
            print(f"Probe diagnostic: distinct offset histories at depth {depth_mm:g} mm.")

    coord_rows = []
    for depth_mm in probe_depths_mm:
        for off, yq in zip(offsets_mm, probe_y_m):
            coord_rows.append([off, probe_x_m*1000.0, yq*1000.0, depth_mm])
    pd.DataFrame(coord_rows, columns=[
        "distance_from_weld_line_mm", "probe_x_mm", "probe_y_mm", "probe_depth_mm"
    ]).to_csv(out/"probe_coordinates.csv", index=False)

    energy = pd.DataFrame(energy_records, columns=[
        "time_s", "maximum_temperature_C", "mean_temperature_C",
        "cumulative_heat_input_J", "cumulative_boundary_loss_J",
        "cumulative_sensible_energy_change_J", "energy_balance_residual_J"
    ])
    energy.to_csv(out/"temperature_summary.csv", index=False)

    # Save final 3D temperature field and coordinate arrays.
    np.savez_compressed(out/"final_temperature_field.npz", T_C=T, x_m=x, y_m=y, z_m=z)

    # Save snapshot top-surface temperatures as CSV for portability.
    for idx, (ts, field) in enumerate(snapshots):
        if idx in (0, len(snapshots)//4, len(snapshots)//2, 3*len(snapshots)//4, len(snapshots)-1):
            pd.DataFrame(field.T, index=y*1000, columns=np.round(x*1000, 3)).to_csv(
                out/f"top_surface_T_{ts:.2f}s.csv", index_label="y_mm"
            )

    # Plot top-surface temperature snapshots.
    selected = sorted(set([0, len(snapshots)//4, len(snapshots)//2, 3*len(snapshots)//4, len(snapshots)-1]))
    fig, axes = plt.subplots(1, len(selected), figsize=(4*len(selected), 3.5), constrained_layout=True)
    if len(selected) == 1:
        axes = [axes]
    for ax, idx in zip(axes, selected):
        ts, field = snapshots[idx]
        im = ax.imshow(field.T, origin="lower", aspect="auto",
                       extent=[x[0]*1000, x[-1]*1000, y[0]*1000, y[-1]*1000],
                       cmap="inferno")
        ax.set_title(f"t = {ts:.1f} s")
        ax.set_xlabel("x (mm)")
        ax.set_ylabel("y (mm)")
    fig.colorbar(im, ax=axes, label="Temperature (°C)", shrink=0.85)
    fig.savefig(out/"top_surface_temperature_snapshots.png", dpi=180)
    plt.close(fig)

    # Plot thermal histories separately for each sampled depth.
    for depth_mm in probe_depths_mm:
        fig, ax = plt.subplots(figsize=(8, 5))
        depth_data = probes[probes.probe_depth_mm == depth_mm]
        for off in offsets_mm:
            sub = depth_data[depth_data.distance_from_weld_line_mm == off]
            ax.plot(sub.time_s, sub.temperature_C, label=f"{off:g} mm")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Temperature (°C)")
        ax.set_title(f"Thermal histories at depth z = {depth_mm:g} mm")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)
        fig.tight_layout()
        fig.savefig(out/f"probe_thermal_histories_z{depth_mm:g}mm.png", dpi=180)
        plt.close(fig)

    summary = {
        "grid_cells": list(T.shape),
        "grid_spacing_m": [dx,dy,dz],
        "probe_extraction": "Trilinear interpolation at exact x/y/z coordinates; no nearest-cell assignment.",
        "probe_depths_mm": probe_depths_mm,
        "bottom_boundary_condition": "Bottom face uses half-cell conduction resistance in series with contact resistance to fixed-temperature steel table.",
        "bottom_contact_conductance_W_m2K": cfg.h_contact,
        "support_table_temperature_C_assumed": cfg.T_table_C,
        "exposed_surface_boundary_condition": "Convection plus radiation on top and four side faces.",
        "probe_x_mm": probe_x_m*1000.0,
        "probe_y_offsets_mm": offsets_mm.tolist(),
        "probe_y_coordinates_mm": (probe_y_m*1000.0).tolist(),
        "probe_z_note": "1.0, 3.75 and 6.75 mm are cell-centred interpolated planes; 7.5 mm is reconstructed at the physical backside face using the bottom conduction/contact resistance model.",
        "interpolation_unit_test": "passed",
        "time_step_s": dt,
        "diffusion_stability_limit_s": dt_diff,
        "source_heating_limit_s": dt_source,
        "number_of_steps": nsteps,
        "simulated_time_s": t_end,
        "peak_temperature_C": float(T.max()),
        "density_model": "Temperature-dependent density interpolated from user-supplied density column.",
        "density_min_kg_m3_over_table": float(np.min(RHO_TABLE)),
        "density_max_kg_m3_over_table": float(np.max(RHO_TABLE)),
        "material_property_valid_temperature_range_C": [float(MAT_T.min()), float(MAT_T.max())],
        "material_property_extrapolation": "Endpoint values are held constant outside 0-650 C; results above 650 C are not supported by supplied property data.",
        "weld_plate_length_mm": cfg.Lx*1000.0,
        "weld_bead_length_mm": cfg.weld_length*1000.0,
        "weld_start_x_mm": cfg.x_start*1000.0,
        "weld_end_x_mm": weld_end_x*1000.0,
        "start_and_end_margin_mm": [cfg.x_start*1000.0, (cfg.Lx-weld_end_x)*1000.0],
        "calculated_weld_time_s": calculated_weld_time,
        "convection_W_m2K_assumed": cfg.h_convection,
        "contact_conductance_W_m2K_assumed": cfg.h_contact,
        "table_temperature_C_assumed": cfg.T_table_C,
        "net_power_W": cfg.efficiency*cfg.voltage*cfg.current,
        "heat_source_model": cfg.source_model,
        "uniform_footprint_length_m": cfg.footprint_length if cfg.source_model == "uniform_surface" else None,
        "uniform_footprint_width_m": cfg.footprint_width if cfg.source_model == "uniform_surface" else None,
        # Source-specific diagnostics: Goldak does not define a surface
        # footprint area or a uniform surface heat flux.
        "uniform_surface_flux_W_m2_at_start": (
            source_field(X, Y, Z, 0.0, cfg, dx, dy, dz)[3].get("surface_flux_W_m2")
            if cfg.source_model == "uniform_surface" else None
        ),
        "uniform_covered_footprint_area_m2_at_start": (
            source_field(X, Y, Z, 0.0, cfg, dx, dy, dz)[3].get("covered_footprint_area_m2")
            if cfg.source_model == "uniform_surface" else None
        ),
        "uniform_nominal_footprint_area_m2": (
            cfg.footprint_length*cfg.footprint_width
            if cfg.source_model == "uniform_surface" else None
        ),
        "heat_input_note": "Exactly 85% efficiency is applied once. Goldak volumetric source is discretely normalised to eta*V*I." if cfg.source_model == "goldak" else "Uniform surface source applies eta*V*I over the specified footprint.",
        "energy_balance_method": "Explicit-step global balance: accumulated sensible storage increment = accumulated applied heat - accumulated boundary losses. Internal conductive face fluxes cancel in the domain sum.",
        "cumulative_heat_input_J": cumulative_input_J,
        "cumulative_boundary_loss_J": cumulative_boundary_loss_J,
        "cumulative_sensible_energy_change_J": cumulative_storage_change_J,
        "final_energy_balance_residual_J": cumulative_storage_change_J - (cumulative_input_J - cumulative_boundary_loss_J)
    }
    with open(out/"run_summary.json","w") as f:
        json.dump(summary,f,indent=2)

    print('Derived density from supplied k, alpha and cp table (kg/m3):')
    print(material_table_diagnostics().to_string(index=False))
    print(json.dumps(summary, indent=2))
    print(f"\nResults saved in: {out.resolve()}")
    return T, probes, summary


if __name__ == "__main__":
    run_simulation()


Probe diagnostic: distinct offset histories at depth 1 mm.
Probe diagnostic: distinct offset histories at depth 3.75 mm.
Probe diagnostic: distinct offset histories at depth 6.75 mm.
Probe diagnostic: distinct offset histories at depth 7.5 mm.
Derived density from supplied k, alpha and cp table (kg/m3):
 temperature_C  k_W_mK  alpha_m2_s  cp_J_kgK  density_supplied_kg_m3  density_from_k_alpha_cp_kg_m3  density_difference_kg_m3
           0.0  51.062    0.000014   571.235             6384.912889                    6384.912889             -2.412298e-07
         100.0  51.062    0.000013   571.235             6797.625890                    6797.625890              3.135192e-07
         200.0  47.324    0.000012   571.235             6915.280782                    6915.280782              2.512097e-07
         300.0  43.840    0.000011   571.235             7060.349375                    7060.349375             -4.352933e-08
         400.0  40.424    0.000010   614.841             6799.077